# Clouds → floods: GOES time-lapse vs. next-day flood polygons

**Research question (open):** do the clouds moving over CONUS on one day line up
with where extreme floods appear the *next* day? This notebook puts three things
on a single scroll-zoom map so you can eyeball that relationship:

1. **A GOES time-lapse** — every daytime frame for a chosen date (6 images,
   16–21 UTC), reprojected to lat/lon and animated with a play/slider.
2. **A 25 × 25 km reference grid** over the CONUS land area — the spatial sampling
   unit we'll likely extract features on later (each cell has a `cell_id`).
3. **The next day's floods** from the unified layer built in
   [`explore_flood_data.ipynb`](explore_flood_data.ipynb) — three toggleable
   sources: observed **groundsource** extents (blue), NWS **flash-flood warnings**
   (red), and **areal-flood warnings** (orange), each active on the target day.

All layers toggle from the **layer control** (top-right). Play/pause + the frame
slider are at the **bottom-left**. Scroll to zoom, drag to pan.

This notebook is **self-contained** — the first cell defines every helper it
needs. The default date is **2021-08-31 (Hurricane Ida)**: watch the storm
sweep the Gulf → Northeast, then toggle on the floods that hit on 09-01.

## Setup & helpers

Run this once. It defines everything below — GOES reprojection, the 25 km
grid, the unified flood loader, and the time-lapse map. Requires the unified
parquet from `explore_flood_data.ipynb`.

In [ ]:
# ---------------------------------------------------------------------------
# Self-contained helpers — run this cell once.
# GOES geostationary imagery → reprojected web-map overlay, a 25 km CONUS land
# grid, the unified flood layer, and a play/slider time-lapse, all on one map.
# ---------------------------------------------------------------------------
import base64
import io
import json
import urllib.request
from datetime import date, datetime, timedelta
from pathlib import Path

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from branca.element import MacroElement
from folium.raster_layers import ImageOverlay
from jinja2 import Template
from PIL import Image
from shapely.geometry import box

# ---- paths (work whether cwd is the repo root or notebooks/) ----
ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "data").exists() else ROOT.parent
DATA_DIR = Path("/mnt/disk1/goes-data")             # GOES NetCDFs
AUX_DIR = DATA_DIR / "aux"                          # cached boundaries / grids
# Unified groundsource + NWS-warning layer (built in explore_flood_data.ipynb)
UNIFIED_PARQUET = ROOT / "data/flood_warnings/floods_unified.parquet"

# CONUS land box (lon_min, lon_max, lat_min, lat_max) — drops AK/HI/PR/territories
CONUS_BBOX = (-125.0, -66.5, 24.0, 50.0)
CONUS_ALBERS = 5070                                 # equal-area metres for the grid

US_STATES_GEOJSON = AUX_DIR / "us-states.geojson"
US_STATES_URL = (
    "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/"
    "master/data/geojson/us-states.json"
)
NON_CONUS_STATES = {"Alaska", "Hawaii", "Puerto Rico"}

# Per-source draw style for the unified flood layer
SOURCE_STYLE = {
    "groundsource": {"label": "groundsource (observed)", "color": "#0b4dd6",
                     "fillColor": "#1f78ff", "fillOpacity": 0.55, "weight": 0.5},
    "ff_warning":   {"label": "flash-flood warnings", "color": "#e31a1c",
                     "fillColor": "#e31a1c", "fillOpacity": 0.15, "weight": 1.0},
    "fa_warning":   {"label": "areal-flood warnings", "color": "#ff7f00",
                     "fillColor": "#ff7f00", "fillOpacity": 0.15, "weight": 1.0},
}

_BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAA"
          "C0lEQVR42mNk+M8AAAMBAQDJ/pLvAAAAAElFTkSuQmCC")  # 1x1 transparent


# ---- file discovery ----
def _scan_token(p):
    """The filename's s{YYYYDDDHHMMSSf} scan-start token."""
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _stamp(p):
    """'YYYY-MM-DD HH:MM UTC' from a filename's scan-start token."""
    t = _scan_token(p)
    if t.startswith("s") and len(t) >= 12:
        d = datetime.strptime(t[1:8], "%Y%j")
        return f"{d:%Y-%m-%d} {t[8:10]}:{t[10:12]} UTC"
    return p.name


def find_files(dt, data_dir=DATA_DIR):
    """All GOES NetCDFs for a date, sorted by scan time (both satellites)."""
    pat = f"*/{dt.year}/{dt.month:02d}/{dt.day:02d}/*.nc"
    return sorted(data_dir.glob(pat), key=_scan_token)


# ---- band access & scaling ----
def cmi(ds, n):
    """The CMI_C<n> band (reflectance or brightness temperature)."""
    return ds[f"CMI_C{n:02d}"]


def _cmap(da):
    bt = str(da.attrs.get("units", "")).strip().upper().startswith("K")
    return "gray_r" if bt else "gray"               # IR: cold cloud tops = white


def _stretch(a, vmin=None, vmax=None):
    lo = np.nanpercentile(a, 2) if vmin is None else vmin
    hi = np.nanpercentile(a, 98) if vmax is None else vmax
    return float(lo), float(hi)


def _norm(a, lo, hi):
    return np.clip((a - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros_like(a)


# ---- crop + reproject geostationary -> lat/lon, then encode an RGBA overlay ----
def _geos(ds):
    return pyproj.CRS.from_cf(dict(ds["goes_imager_projection"].attrs))


def crop_lonlat(ds, lon_min, lon_max, lat_min, lat_max):
    """Subset to a lon/lat box (degrees) before reprojecting — faster + zoomed."""
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    tf = pyproj.Transformer.from_crs("EPSG:4326", _geos(ds), always_xy=True)
    xs, ys = tf.transform([lon_min, lon_max, lon_min, lon_max],
                          [lat_min, lat_min, lat_max, lat_max])
    xs, ys = np.array(xs) / h, np.array(ys) / h
    xs, ys = xs[np.isfinite(xs)], ys[np.isfinite(ys)]
    if xs.size == 0 or ys.size == 0:
        raise ValueError("box is off the Earth disk for this satellite")
    return ds.sel(x=slice(xs.min(), xs.max()), y=slice(ys.max(), ys.min()))


def reproject_bands(ds, bands):
    """Reproject bands to EPSG:4326; return (DataArray[band], [[s, w], [n, e]])."""
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    da = xr.concat([cmi(ds, n).reset_coords(drop=True) for n in bands], dim="band")
    da = da.assign_coords(x=ds["x"] * h, y=ds["y"] * h)
    da = da.rio.write_crs(_geos(ds)).rio.set_spatial_dims(x_dim="x", y_dim="y")
    da = da.rio.reproject("EPSG:4326", nodata=np.nan)
    da = da.sortby("y", ascending=False).sortby("x")
    minx, miny, maxx, maxy = da.rio.bounds()
    return da, [[miny, minx], [maxy, maxx]]


def _rgba_single(a, cmap, lo, hi):
    finite = np.isfinite(a)
    sm = plt.cm.ScalarMappable(norm=plt.Normalize(lo, hi), cmap=cmap)
    rgba = sm.to_rgba(np.nan_to_num(a, nan=lo), bytes=True)
    rgba[..., 3] = np.where(finite, 255, 0).astype("uint8")
    return rgba


def _rgba_rgb(arr3, gamma=2.2):
    chans, alpha = [], np.ones(arr3.shape[1:], dtype=bool)
    for a in arr3:
        chans.append(_norm(a, *_stretch(a)))
        alpha = alpha & np.isfinite(a)
    rgb = np.nan_to_num(np.clip(np.dstack(chans) ** (1 / gamma), 0, 1))
    a8 = np.where(alpha, 255, 0).astype("uint8")
    return np.dstack([(rgb * 255).astype("uint8"), a8])


def _data_uri(rgba):
    buf = io.BytesIO()
    Image.fromarray(rgba, mode="RGBA").save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def _downsample(rgba, max_px):
    """Shrink an RGBA frame so the embedded HTML stays small."""
    h, w = rgba.shape[:2]
    if max(h, w) <= max_px:
        return rgba
    s = max_px / max(h, w)
    im = Image.fromarray(rgba, "RGBA").resize(
        (max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    return np.asarray(im)


def _frame_rgba(ds, band, rgb, crop):
    """Reproject one dataset to an RGBA array + lat/lon bounds for an overlay."""
    sub = crop_lonlat(ds, *crop) if crop else ds
    if rgb is not None:
        da_ll, bounds = reproject_bands(sub, list(rgb))
        return _rgba_rgb(da_ll.values), bounds
    da_ll, bounds = reproject_bands(sub, [band])
    a = da_ll.isel(band=0).values
    return _rgba_single(a, _cmap(cmi(sub, band)), *_stretch(a)), bounds


# ---- 25 km CONUS land grid ----
def conus_land(states_geojson=US_STATES_GEOJSON):
    """CONUS land polygon (lower-48 + DC) in EPSG:4326 (downloads once if missing)."""
    if not states_geojson.exists():
        states_geojson.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading US states boundary -> {states_geojson}")
        urllib.request.urlretrieve(US_STATES_URL, states_geojson)
    g = gpd.read_file(states_geojson)
    return g[~g["name"].isin(NON_CONUS_STATES)].to_crs(4326)


def conus_grid(cell_km=25.0, cache=True):
    """Fishnet of cell_km square cells over CONUS land (EPSG:4326, cached).

    Cells are built in CONUS Albers (EPSG:5070) so they are genuinely cell_km on
    a side, then kept where they intersect land. Each row has a cell_id — the
    natural sampling unit for later feature extraction.
    """
    cache_path = AUX_DIR / f"conus_grid_{int(cell_km)}km.parquet"
    if cache and cache_path.exists():
        return gpd.read_parquet(cache_path)
    land = conus_land().to_crs(CONUS_ALBERS)
    land_union = land.union_all()
    step = cell_km * 1000.0
    minx, miny, maxx, maxy = land.total_bounds
    cells, ids = [], []
    for r, y0 in enumerate(np.arange(np.floor(miny / step) * step, maxy + step, step)):
        for c, x0 in enumerate(np.arange(np.floor(minx / step) * step, maxx + step, step)):
            cells.append(box(x0, y0, x0 + step, y0 + step))
            ids.append(f"r{r:03d}c{c:03d}")
    grid = gpd.GeoDataFrame({"cell_id": ids}, geometry=cells, crs=CONUS_ALBERS)
    grid = grid[grid.intersects(land_union)].reset_index(drop=True).to_crs(4326)
    if cache:
        AUX_DIR.mkdir(parents=True, exist_ok=True)
        grid.to_parquet(cache_path)
    return grid


# ---- unified flood layer (groundsource + FF/FA warnings) ----
def load_unified(target, bbox=CONUS_BBOX, sources=None, parquet=UNIFIED_PARQUET):
    """Unified flood polygons active on `target` within `bbox`.

    Built in explore_flood_data.ipynb (source, event_id, phenomena, issue_date,
    expire_date, area_km2, geometry). "Active" = issue_date <= target <= expire_date.
    Pass `sources` to keep only some of groundsource / ff_warning / fa_warning.
    """
    if not parquet.exists():
        raise FileNotFoundError(
            f"{parquet} not found — run the 'Persist the unified frame' cell in "
            "explore_flood_data.ipynb first.")
    ts = pd.Timestamp(target)
    g = gpd.read_parquet(parquet)
    g = g[(g["issue_date"] <= ts + pd.Timedelta(days=1)) & (g["expire_date"] >= ts)]
    if sources is not None:
        g = g[g["source"].isin(sources)]
    lon_min, lon_max, lat_min, lat_max = bbox
    return g.cx[lon_min:lon_max, lat_min:lat_max].reset_index(drop=True)


# ---- folium layers ----
def grid_layer(grid, name="25 km grid"):
    """Thin, unfilled grid outlines as a toggleable GeoJson layer."""
    return folium.GeoJson(
        grid[["cell_id", "geometry"]].to_json(), name=name,
        style_function=lambda _f: {"color": "#444", "weight": 0.4,
                                   "fill": False, "opacity": 0.5},
        tooltip=folium.GeoJsonTooltip(fields=["cell_id"], aliases=["cell"]))


def unified_layers(floods):
    """One toggleable GeoJson layer per source (blue extents, red/orange warnings)."""
    layers = []
    for src, style in SOURCE_STYLE.items():
        sub = floods[floods["source"] == src]
        if not len(sub):
            continue
        gj = sub[["event_id", "phenomena", "issue_date", "expire_date",
                  "area_km2", "geometry"]].copy()
        gj["issue_date"] = gj["issue_date"].astype(str)
        gj["expire_date"] = gj["expire_date"].astype(str)
        sf = {k: style[k] for k in ("color", "fillColor", "fillOpacity", "weight")}
        layers.append(folium.GeoJson(
            gj.to_json(), name=f"{style['label']} ({len(sub):,})",
            style_function=lambda _f, sf=sf: sf,
            tooltip=folium.GeoJsonTooltip(
                fields=["phenomena", "issue_date", "expire_date", "area_km2"],
                aliases=["type", "issue", "expire", "km2"])))
    return layers


def _base_map(bounds):
    """Scroll-zoom folium map with OSM / light / satellite basemaps."""
    (s, w), (n, e) = bounds
    m = folium.Map(location=[(s + n) / 2, (w + e) / 2], zoom_start=5,
                   tiles="OpenStreetMap")
    folium.TileLayer("CartoDB positron", name="light").add_to(m)
    folium.TileLayer(
        tiles=("https://server.arcgisonline.com/ArcGIS/rest/services/"
               "World_Imagery/MapServer/tile/{z}/{y}/{x}"),
        attr="Esri World Imagery", name="satellite").add_to(m)
    return m


# ---- time-lapse control (bottom-left play/pause + slider over an ImageOverlay) ----
class _TimeLapse(MacroElement):
    """A bottom-left play/slider that cycles an ImageOverlay through frames."""

    _template = Template("""
        {% macro script(this, kwargs) %}
        (function(){
          var fr={{this.frames_json}}, lb={{this.labels_json}};
          var ov={{this.overlay_name}}, mp={{this._parent.get_name()}};
          var nm="{{this.get_name()}}", iv={{this.interval}}, i=0, t=null, pl=false;
          var c=L.control({position:'bottomleft'});
          c.onAdd=function(){
            var d=L.DomUtil.create('div','');
            d.style.cssText='background:rgba(255,255,255,.88);padding:6px 8px;'
              +'font:12px sans-serif;border-radius:4px';
            d.innerHTML='<button id="b_'+nm+'">&#9658;</button> '
              +'<input id="s_'+nm+'" type="range" min="0" max="'+(fr.length-1)
              +'" value="0" style="width:240px;vertical-align:middle"> '
              +'<span id="l_'+nm+'"></span>';
            L.DomEvent.disableClickPropagation(d);
            return d;
          };
          c.addTo(mp);
          function show(k){i=(k+fr.length)%fr.length;ov.setUrl(fr[i]);
            document.getElementById('s_'+nm).value=i;
            document.getElementById('l_'+nm).innerText=lb[i];}
          function stop(){pl=false;document.getElementById('b_'+nm).innerHTML='&#9658;';
            clearInterval(t);}
          function go(){pl=true;
            document.getElementById('b_'+nm).innerHTML='&#10074;&#10074;';
            t=setInterval(function(){show(i+1);},iv);}
          setTimeout(function(){
            show(0);
            document.getElementById('b_'+nm).onclick=function(){pl?stop():go();};
            document.getElementById('s_'+nm).oninput=function(e){stop();
              show(parseInt(e.target.value));};
            go();
          },300);
        })();
        {% endmacro %}
    """)

    def __init__(self, overlay_name, frames, labels, interval=800):
        super().__init__()
        self._name = "TimeLapse"
        self.overlay_name = overlay_name
        self.frames_json = json.dumps(frames)
        self.labels_json = json.dumps(labels)
        self.interval = interval


# ---- the combined view: clouds (time-lapse) + 25 km grid + next-day floods ----
def flood_cloud_timelapse(goes_date, band=13, rgb=None, crop=None, show_grid=True,
                          grid_cell_km=25.0, flood_lag_days=1, flood_sources=None,
                          max_px=1400, interval=800, opacity=0.8, data_dir=DATA_DIR):
    """Watch a day's GOES clouds against the floods they may have caused.

    Time-lapses every GOES frame on goes_date (6 daytime images), overlays the
    25 km CONUS land grid, and draws the unified flood layer (groundsource +
    FF/FA warnings) active flood_lag_days later — each source a toggleable layer.
    band=N (13 = cold cloud tops, 8 = water vapour) or rgb=(2, 3, 1) for true colour.
    crop=(lon_min, lon_max, lat_min, lat_max) focuses + speeds the view.
    """
    files = find_files(goes_date, data_dir)
    if not files:
        raise FileNotFoundError(f"No GOES files for {goes_date} under {data_dir}")
    print(f"{len(files)} cloud frame(s) on {goes_date}; reprojecting...")

    frames, labels, bounds = [], [], None
    for j, p in enumerate(files):
        ds = xr.open_dataset(p, decode_times=False)
        try:
            rgba, b = _frame_rgba(ds, band, rgb, crop)
        finally:
            ds.close()
        bounds = bounds or b
        frames.append(_data_uri(_downsample(rgba, max_px)))
        labels.append(_stamp(p))
        print(f"  {j + 1}/{len(files)}  {labels[-1]}", end="\r")
    print()

    target = goes_date + timedelta(days=flood_lag_days)
    view_bbox = crop if crop else CONUS_BBOX
    floods = load_unified(target, bbox=view_bbox, sources=flood_sources)
    print(f"{len(floods)} flood polygon(s) active on {target} "
          f"by source: {floods.source.value_counts().to_dict()}")

    m = _base_map(bounds)
    ov = ImageOverlay(_BLANK, bounds=bounds, opacity=opacity,
                      name=f"GOES clouds {goes_date}")
    ov.add_to(m)
    if show_grid:
        lon_min, lon_max, lat_min, lat_max = view_bbox
        grid = conus_grid(grid_cell_km).cx[lon_min:lon_max, lat_min:lat_max]
        grid_layer(grid, f"{int(grid_cell_km)} km grid").add_to(m)
    for layer in unified_layers(floods):
        layer.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds(bounds)
    m.add_child(_TimeLapse(ov.get_name(), frames, labels, interval))
    return m

## The combined view

`flood_cloud_timelapse(goes_date, ...)` reprojects each frame, clips the 25 km
grid and the next-day unified floods (groundsource + FF/FA warnings) to the view,
and returns the map — each flood source is its own toggleable layer. Band **13** (clean IR) makes
cold storm-cloud tops bright — good for tracking the system. First build for a date
is a little slow (it reprojects 6 frames).

> Cropped to the Gulf → Northeast corridor so the embedded map stays light. Pass
> `crop=None` for the full CONUS frame.

In [ ]:
m = flood_cloud_timelapse(
    date(2021, 8, 31),          # Hurricane Ida
    band=13,                    # clean IR: cold cloud tops = bright
    crop=None,    # lon_min, lon_max, lat_min, lat_max
    flood_lag_days=1,           # floods active the *next* day
)
m

## True-colour variant

Same machinery with `rgb=(2, 3, 1)` for a daytime true-colour view — the swirl of
the cyclone reads more naturally, though only the lit part of each frame shows.

In [ ]:
flood_cloud_timelapse(
    date(2021, 8, 31), rgb=(2, 3, 1), crop=None,
)

## The pieces, on their own

The same building blocks are exposed directly, so you can inspect or reuse them:

- `conus_grid(25)` → a cached `GeoDataFrame` of 25 km land cells (`cell_id`, geometry).
- `load_unified(date)` → groundsource + FF/FA warning polygons active on that date.
- `find_files(date)` → the GOES NetCDFs for a day.

In [ ]:
grid = conus_grid(25)
floods = load_unified(date(2021, 9, 1))
print(f"grid: {len(grid):,} cells of 25 km")
print(f"floods active 2021-09-01: {len(floods):,} polygons")
print("by source:", floods.source.value_counts().to_dict())
floods.head(3)

## Try another event

Dates with the most next-day CONUS flooding (good candidates to explore):

| GOES date | next-day floods | total km² |
|---|---|---|
| 2021-08-31 | 2,567 | 180k | ← Hurricane Ida (default) |
| 2024-09-26 | 1,675 | 224k | ← Hurricane Helene |
| 2023-12-17 | 1,669 | 223k |
| 2024-01-08 | 1,589 | 216k |
| 2024-08-08 | 1,573 | 290k |

Swap the date below. Use `band=8` (water vapour) to see moisture transport, or
widen `crop` for a bigger region. Set `crop=None` for all of CONUS (slower, larger).

In [ ]:
flood_cloud_timelapse(
    date(2024, 9, 1),          # Hurricane Helene
    band=13,
    crop=None
)

## Lightning overlay (GLM flashes)

The full combined view with **lightning in the loop**: the same GOES cloud
time-lapse and next-day flood layers (groundsource + NWS warnings), plus the
**GLM flashes within ±30 min of each frame as yellow dots** that advance with
the play/slider — the bottom-left label shows the frame time and its ⚡ count.
The dot layer toggles from the layer control like everything else.

Flash data comes from the per-day parquets under `/mnt/disk1/glm-data`, built by
`src/download_glm.py` — the full 2019–2026 range is still downloading, so the
cell picks a **random already-built day** (re-run for another, or pin
`day = date(...)`). Watch for flash clusters riding under the coldest (brightest
IR) cloud tops — deep convection is a candidate precursor signal for the floods
that show up the next day.

In [ ]:
# ---------------------------------------------------------------------------
# Clouds + floods + GLM lightning, one synced time-lapse.
# Flash dots follow the play/slider: each GOES frame shows the flashes that
# occurred within +/-30 min of its scan. Data: per-day flash parquets built by
# src/download_glm.py (/mnt/disk1/glm-data, growing while the download runs).
# ---------------------------------------------------------------------------
import random

GLM_DIR = Path("/mnt/disk1/glm-data")
FLASH_HALF_WINDOW = pd.Timedelta(minutes=30)    # flashes shown around each frame
MAX_DOTS_PER_FRAME = 5000                       # sampled above this, for the HTML


def glm_days(glm_dir=GLM_DIR):
    """Dates with a built GLM flash parquet (sorted)."""
    return sorted(datetime.strptime(p.stem[-8:], "%Y%m%d").date()
                  for p in glm_dir.glob("*/glm_flashes_*.parquet"))


def load_glm(dt, bbox=CONUS_BBOX, glm_dir=GLM_DIR):
    """One day of GLM flashes clipped to bbox (times, lat/lon, energy, area)."""
    df = pd.read_parquet(glm_dir / str(dt.year) / f"glm_flashes_{dt:%Y%m%d}.parquet")
    lon_min, lon_max, lat_min, lat_max = bbox
    return df[df["lon"].between(lon_min, lon_max)
              & df["lat"].between(lat_min, lat_max)].reset_index(drop=True)


def _frame_time(p):
    """Scan-start datetime from a GOES filename's s{YYYYDDDHHMM...} token."""
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


class _FlashCloudLapse(MacroElement):
    """Play/slider that cycles cloud frames AND that frame's flash dots."""

    _template = Template("""
        {% macro script(this, kwargs) %}
        (function(){
          var fr={{this.frames_json}}, lb={{this.labels_json}}, fl={{this.flashes_json}};
          var ov={{this.overlay_name}}, gp={{this.group_name}};
          var mp={{this._parent.get_name()}};
          var nm="{{this.get_name()}}", iv={{this.interval}}, i=0, t=null, pl=false;
          var cv=L.canvas({padding:0.2});
          var c=L.control({position:'bottomleft'});
          c.onAdd=function(){
            var d=L.DomUtil.create('div','');
            d.style.cssText='background:rgba(255,255,255,.88);padding:6px 8px;'
              +'font:12px sans-serif;border-radius:4px';
            d.innerHTML='<button id="b_'+nm+'">&#9658;</button> '
              +'<input id="s_'+nm+'" type="range" min="0" max="'+(fr.length-1)
              +'" value="0" style="width:240px;vertical-align:middle"> '
              +'<span id="l_'+nm+'"></span>';
            L.DomEvent.disableClickPropagation(d);
            return d;
          };
          c.addTo(mp);
          function dots(k){gp.clearLayers();
            fl[k].forEach(function(p){
              L.circleMarker([p[0],p[1]],{renderer:cv,radius:2.2,weight:.6,
                color:'#8a5a00',fillColor:'#ffd400',fillOpacity:.85}).addTo(gp);});}
          function show(k){i=(k+fr.length)%fr.length;ov.setUrl(fr[i]);dots(i);
            document.getElementById('s_'+nm).value=i;
            document.getElementById('l_'+nm).innerText=
              lb[i]+' \\u00b7 \\u26a1 '+fl[i].length;}
          function stop(){pl=false;document.getElementById('b_'+nm).innerHTML='&#9658;';
            clearInterval(t);}
          function go(){pl=true;
            document.getElementById('b_'+nm).innerHTML='&#10074;&#10074;';
            t=setInterval(function(){show(i+1);},iv);}
          setTimeout(function(){
            show(0);
            document.getElementById('b_'+nm).onclick=function(){pl?stop():go();};
            document.getElementById('s_'+nm).oninput=function(e){stop();
              show(parseInt(e.target.value));};
            go();
          },300);
        })();
        {% endmacro %}
    """)

    def __init__(self, overlay_name, group_name, frames, labels, flash_pts,
                 interval=800):
        super().__init__()
        self._name = "FlashCloudLapse"
        self.overlay_name = overlay_name
        self.group_name = group_name
        self.frames_json = json.dumps(frames)
        self.labels_json = json.dumps(labels)
        self.flashes_json = json.dumps(flash_pts)
        self.interval = interval


def flood_cloud_lightning_timelapse(goes_date, band=13, rgb=None, crop=None,
                                    show_grid=True, grid_cell_km=25.0,
                                    flood_lag_days=1, flood_sources=None,
                                    max_px=1400, interval=800, opacity=0.8,
                                    data_dir=DATA_DIR,
                                    max_dots=MAX_DOTS_PER_FRAME):
    """The combined view plus lightning: clouds, grid, next-day floods, and the
    GLM flashes within +/-30 min of each frame as yellow dots that advance with
    the play/slider. Same knobs as flood_cloud_timelapse."""
    files = find_files(goes_date, data_dir)
    if not files:
        raise FileNotFoundError(f"No GOES files for {goes_date} under {data_dir}")
    view_bbox = crop if crop else CONUS_BBOX
    flashes = load_glm(goes_date, bbox=view_bbox)
    print(f"{len(files)} cloud frame(s) on {goes_date}, "
          f"{len(flashes):,} GLM flashes in view; reprojecting...")

    frames, labels, flash_pts, bounds = [], [], [], None
    for j, p in enumerate(files):
        ds = xr.open_dataset(p, decode_times=False)
        try:
            rgba, b = _frame_rgba(ds, band, rgb, crop)
        finally:
            ds.close()
        bounds = bounds or b
        frames.append(_data_uri(_downsample(rgba, max_px)))
        labels.append(_stamp(p))
        t = _frame_time(p)
        win = flashes[(flashes["time_start"] >= t - FLASH_HALF_WINDOW)
                      & (flashes["time_start"] < t + FLASH_HALF_WINDOW)]
        if len(win) > max_dots:
            win = win.sample(max_dots, random_state=0)
        flash_pts.append(win[["lat", "lon"]].astype(float).round(3).values.tolist())
        print(f"  {j + 1}/{len(files)}  {labels[-1]}  ({len(win):,} flashes)",
              end="\r")
    print()

    target = goes_date + timedelta(days=flood_lag_days)
    floods = load_unified(target, bbox=view_bbox, sources=flood_sources)
    print(f"{len(floods)} flood polygon(s) active on {target} "
          f"by source: {floods.source.value_counts().to_dict()}")

    m = _base_map(bounds)
    ov = ImageOverlay(_BLANK, bounds=bounds, opacity=opacity,
                      name=f"GOES clouds {goes_date}")
    ov.add_to(m)
    if show_grid:
        lon_min, lon_max, lat_min, lat_max = view_bbox
        grid = conus_grid(grid_cell_km).cx[lon_min:lon_max, lat_min:lat_max]
        grid_layer(grid, f"{int(grid_cell_km)} km grid").add_to(m)
    for layer in unified_layers(floods):
        layer.add_to(m)
    n_shown = sum(len(p) for p in flash_pts)
    fg = folium.FeatureGroup(name=f"GLM flashes ±30 min of frame "
                                  f"({n_shown:,} dots)")
    fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds(bounds)
    m.add_child(_FlashCloudLapse(ov.get_name(), fg.get_name(), frames, labels,
                                 flash_pts, interval))
    return m


day = random.choice(glm_days())         # or pin one: day = date(2019, 5, 20)
flood_cloud_lightning_timelapse(day, band=13, flood_lag_days=1)

## Notes & next steps

- **Grid is the sampling unit.** Each 25 km `cell_id` is a candidate location for
  per-cell features (cloud-top temperature, water-vapour, motion) vs. a next-day
  flood label. The grid is cached at `/mnt/disk1/goes-data/aux/conus_grid_25km.parquet`.
- **"Next day" is configurable** via `flood_lag_days` — try 0 (same day) or 2.
- **Daytime only.** We pull 16–21 UTC frames, so true-colour is partial near the
  edges; IR (band 13) and water-vapour (band 8) work day and night.
- **A flood polygon's `start_date` is when it was first observed**, which may lag
  the rain by a day or two — keep that in mind when reading the overlap.